In [2]:
from astropy.table import Table
t = Table.read("/Users/ujjwalsingh/Documents/pyscope/tests/bin/schedule.ecsv")
t.show_in_notebook()

         interactive tables it is recommended to use dedicated tools like:
         - https://github.com/bloomberg/ipydatagrid
         - https://docs.bokeh.org/en/latest/docs/user_guide/interaction/widgets.html#datatable
         - https://dash.plotly.com/datatable [warnings]


idx,ID,name,priority,observer,code,title,filename,filter,exposure,nexp,target_ra,target_dec,start_time,end_time,duration,pm_ra_cosdec,pm_dec,status,message
,,,,,,,,,s,,deg,deg,,,s,arcsec / h,arcsec / h,,
0,60810.6824608532,Deneb_B,0,observer1,test,Deneb B-band Observation,--,B,60.0,16,310.357979753,45.280338807,2016-07-07T02:00:00.000,2016-07-07T02:21:20.000,1280.0,0.0,0.0,S,Scheduled
1,60810.682960858256,M13_B,0,observer1,test,M13 B-band Observation,--,B,100.0,16,250.423475,36.46131944,2016-07-07T02:21:50.000,2016-07-07T02:53:50.000,1920.0,0.0,0.0,S,Scheduled
2,60810.682460862976,Deneb_G,1,observer1,test,Deneb G-band Observation,--,G,60.0,16,310.357979753,45.280338807,2016-07-07T02:54:00.000,2016-07-07T03:15:20.000,1280.0,0.0,0.0,S,Scheduled
3,60810.68396086508,M13_G,1,observer1,test,M13 G-band Observation,--,G,100.0,16,250.423475,36.46131944,2016-07-07T03:15:50.000,2016-07-07T03:47:50.000,1920.0,0.0,0.0,S,Scheduled
4,60810.68246086786,Deneb_R,2,observer1,test,Deneb R-band Observation,--,R,60.0,16,310.357979753,45.280338807,2016-07-07T03:48:00.000,2016-07-07T04:09:20.000,1280.0,0.0,0.0,S,Scheduled
5,60810.684960869854,M13_R,2,observer1,test,M13 R-band Observation,--,R,100.0,16,250.423475,36.46131944,2016-07-07T04:09:50.000,2016-07-07T04:41:50.000,1920.0,0.0,0.0,S,Scheduled


In [42]:
from pyscope.telrun import sch
from astropy.table import Table

# t = sch.read('/Users/ujjwalsingh/Documents/pyscope/tests/bin/test_sch.sch')
t = sch.read('/Users/ujjwalsingh/Documents/pyscope/tests/bin/valid/alc.sch')

my_table = Table(t)
# my_table.write("my_data.ecsv")


In [50]:
import astropy.units as u
from astropy.time import Time
from astroplan import ObservingBlock, FixedTarget
from astropy.coordinates import SkyCoord

def dict_blocks_to_observing_blocks(dict_blocks):
    """
    Convert a list of pyscope observation dictionaries to astroplan.ObservingBlock objects.
    
    Parameters
    ----------
    dict_blocks : list of dict
        List of observation dictionaries with pyscope/schedtel structure
        
    Returns
    -------
    observing_blocks : list of astroplan.ObservingBlock
        List of astroplan ObservingBlock objects
    global_constraints : list
        List of constraints that were extracted from the dictionaries
    """
    observing_blocks = []
    all_constraints = []
    
    for block_dict in dict_blocks:
        # Extract target information
        if 'target' in block_dict and block_dict['target'] is not None:
            target = block_dict['target']
        elif 'target_ra' in block_dict and 'target_dec' in block_dict:
            # Create target from RA/Dec if target object not available
            target = FixedTarget(
                coord=SkyCoord(
                    ra=block_dict['target_ra'] * u.deg,
                    dec=block_dict['target_dec'] * u.deg
                ),
                name=block_dict.get('name', 'Unknown')
            )
        else:
            raise ValueError(f"Block missing target information: {block_dict.get('name', 'Unknown block')}")
        
        # Extract duration - ensure it has units
        if 'duration' in block_dict:
            duration = block_dict['duration']
            if not hasattr(duration, 'unit'):
                # Assume seconds if no units
                duration = duration * u.second
        else:
            # Calculate duration from exposure time and number of exposures
            exp_time = block_dict.get('exposure', 1) * u.second
            n_exp = block_dict.get('nexp', 1)
            readout_time = block_dict.get('readout', 0) * u.second
            duration = (exp_time + readout_time) * n_exp
        
        # Extract priority
        priority = block_dict.get('priority', 1)
        
        # Create configuration dictionary from relevant block parameters
        configuration = {}
        config_keys = ['filter', 'exposure', 'nexp', 'binning', 'frame_position', 
                      'frame_size', 'repositioning', 'shutter_state', 'readout', 
                      'type', 'backend', 'code', 'observer', 'title']
        
        for key in config_keys:
            if key in block_dict:
                configuration[key] = block_dict[key]
        
        # Extract constraints for this block
        block_constraints = block_dict.get('constraints', None)
        if block_constraints is not None:
            all_constraints.extend(block_constraints)
        
        # Extract name/ID
        name = block_dict.get('name', block_dict.get('ID', 'Unknown'))
        
        # Create ObservingBlock
        obs_block = ObservingBlock(
            target=target,
            duration=duration,
            priority=priority,
            configuration=configuration,
            constraints=block_constraints,
            name=str(name)
        )
        
        # Store additional metadata that might be useful
        obs_block._pyscope_metadata = {
            'original_dict': block_dict,
            'ID': block_dict.get('ID'),
            'status': block_dict.get('status'),
            'message': block_dict.get('message'),
            'start_time': block_dict.get('start_time'),
            'end_time': block_dict.get('end_time'),
            'sched_time': block_dict.get('sched_time'),
            'filename': block_dict.get('filename', ''),
            'comment': block_dict.get('comment', ''),
            'sch': block_dict.get('sch', ''),
            'pm_ra_cosdec': block_dict.get('pm_ra_cosdec'),
            'pm_dec': block_dict.get('pm_dec')
        }
        
        observing_blocks.append(obs_block)
    
    # Remove duplicate constraints and create a global constraints list
    unique_constraints = []
    for constraint in all_constraints:
        if constraint not in unique_constraints:
            unique_constraints.append(constraint)
    
    return observing_blocks, unique_constraints

In [62]:
o, u = dict_blocks_to_observing_blocks(t)

In [73]:
o[0]

<astroplan.scheduling.ObservingBlock (icrs, unscheduled) at 0x142ccb530>

In [72]:
t[0]

{'ID': np.float64(60828.23948169229),
 'name': 'M57',
 'priority': 1,
 'observer': ['leecarkner@augustana.edu'],
 'code': 'alc',
 'title': 'M57 obs for astr-145',
 'filename': '',
 'type': 'light',
 'backend': 0,
 'filter': 'g',
 'exposure': 30.0,
 'nexp': 1,
 'repositioning': (0, 0),
 'shutter_state': True,
 'readout': 0,
 'binning': (2, 2),
 'frame_position': (0, 0),
 'frame_size': (0, 0),
 'pm_ra_cosdec': <Quantity 0. arcsec / h>,
 'pm_dec': <Quantity 0. arcsec / h>,
 'comment': '',
 'sch': 'alc',
 'status': 'U',
 'message': 'Unscheduled',
 'sched_time': None,
 'duration': <Quantity 30. s>,
 'target': <SkyCoord (ICRS): (ra, dec) in deg
     (283.39623652, 33.02913425)>,
 'target_ra': np.float64(283.39623652463),
 'target_dec': np.float64(33.02913424654),
 'constraints': None,
 'start_time': None,
 'end_time': None}

## sch.py ecsv implementation

In [28]:
from pyscope.telrun import sch

obs_list = sch.read('/Users/ujjwalsingh/Documents/pyscope/tests/bin/xpg002.sch')

obs_table = sch.create_ecsv_table(obs_list)
sch.save_schedule_to_ecsv(obs_list, "example_schedule.ecsv")



ID,name,priority,observer,code,title,filename,filter,exposure,nexp,target_ra,target_dec,start_time,end_time,duration,pm_ra_cosdec,pm_dec,status,message
,,,,,,,,s,,deg,deg,,,s,arcsec / h,arcsec / h,,
float64,str3,int64,str24,str3,str15,str1,str3,float64,int64,float64,float64,str1,str1,float64,float64,float64,str1,str11
60810.34965219702,m51,1,philip-griffin@uiowa.edu,xpg,Schedtel test 3,,lum,15.0,1,202.469575,47.1952583,,,15.0,0.0,0.0,U,Unscheduled
60810.34965220061,m51,1,philip-griffin@uiowa.edu,xpg,Schedtel test 3,,lum,15.0,1,202.469575,47.1952583,,,15.0,0.0,0.0,U,Unscheduled
60810.349652201796,m51,1,philip-griffin@uiowa.edu,xpg,Schedtel test 3,,lum,15.0,1,202.469575,47.1952583,,,15.0,0.0,0.0,U,Unscheduled
60810.34965220278,m51,1,philip-griffin@uiowa.edu,xpg,Schedtel test 3,,lum,15.0,1,202.469575,47.1952583,,,15.0,0.0,0.0,U,Unscheduled
60810.349652203644,m51,1,philip-griffin@uiowa.edu,xpg,Schedtel test 3,,lum,15.0,1,202.469575,47.1952583,,,15.0,0.0,0.0,U,Unscheduled
60810.34965220448,m51,1,philip-griffin@uiowa.edu,xpg,Schedtel test 3,,lum,15.0,1,202.469575,47.1952583,,,15.0,0.0,0.0,U,Unscheduled
60810.34965220528,m51,1,philip-griffin@uiowa.edu,xpg,Schedtel test 3,,lum,15.0,1,202.469575,47.1952583,,,15.0,0.0,0.0,U,Unscheduled
60810.349652206045,m51,1,philip-griffin@uiowa.edu,xpg,Schedtel test 3,,lum,15.0,1,202.469575,47.1952583,,,15.0,0.0,0.0,U,Unscheduled


In [29]:
schedule_table = Table.read('example_schedule.ecsv', format='ascii.ecsv')
from pyscope.telrun import schedtab
observing_blocks = schedtab.table_to_blocks(schedule_table)

print(observing_blocks)


KeyError: 'constraints'

# Trying to run schtedtel.py

In [37]:
from pyscope.telrun import schedtel
from pyscope.telrun import plot_schedule_gantt, plot_schedule_sky, schedtel


catalog = "/Users/ujjwalsingh/Documents/pyscope/tests/bin/test_schedtel.cat"
observatory = "/Users/ujjwalsingh/Documents/pyscope/tests/bin/simulator_observatory.cfg"

schedule = schedtel(
        catalog=catalog,
        observatory=observatory,
        filename= "test_schedtel.ecsv",
    )

Type of block_groups: <class 'list'>
Type of block_groups[0]: <class 'list'>


  0%|          | 0/4 [00:00<?, ?it/s]

Error in scheduler: list index out of range


AttributeError: 'NoneType' object has no attribute 'calc_reconfig_time_blocks'

In [38]:
from astroplan import Observer, FixedTarget
from astropy.time import Time
from astropy import units as u
from astroplan.constraints import AirmassConstraint, AtNightConstraint, TimeConstraint, AltitudeConstraint
from pyscope.telrun.create_priority_schedule import create_priority_schedule  # replace with your actual import

# 1. Define your targets and (optional) names
targets = [
    FixedTarget.from_name('Deneb'),
    FixedTarget.from_name('M13'),
]


# 2. Create an Observer
observer = Observer.at_site('Rubin')

# 3. Define your time window
start_time      = Time('2025-07-06 19:00')  # UT
end_time        = Time('2025-07-07 19:00')  # UT

# 6 h half‑night sub‑window
half_night_start = Time('2025-07-07 02:00')  # UT
half_night_end   = Time('2025-07-07 08:00')  # UT

# 4. (Optional) per‑target priorities and durations
  # lower number = schedule first
targets = [FixedTarget.from_name('Deneb'), FixedTarget.from_name('M13'), 
                   FixedTarget.from_name('Vega'), 
                   FixedTarget.from_name('Polaris'), 
                   FixedTarget.from_name('Altair'), 
                   FixedTarget.from_name('Albireo'),
                   FixedTarget.from_name('Arcturus'),FixedTarget.from_name('Capella')]
durations = [5*u.minute, 5*u.minute, 5*u.minute, 5*u.minute, 5*u.minute, 5*u.minute, 5*u.minute, 5*u.minute]
constraints = []
configuration = [{'filter': 'B'},{'filter': 'B'},{'filter': 'B'},{'filter': 'B'},{'filter': 'B'},{'filter': 'B'},{'filter': 'B'},{'filter': 'B'}]

# 8. Call your function
schedule, scheduler = create_priority_schedule(
    targets=targets,
    observer=observer,
    start_time=start_time,
    end_time=end_time,
    durations=durations,
    constraints=constraints,
    configuration= configuration
    
)

# 9. Inspect the schedule
from astropy.table import Table
from astroplan.scheduling import TransitionBlock


Arcturus = schedule.observing_blocks[0]
print("the priority right now is ", Arcturus.priority) # will print current priority
Arcturus.priority = 10 # set it to 10, if the current test requires a lower priority relative to another star etc 


blocks = [Arcturus]
print(" these are the obs blocks", blocks, "the priority is changed to ", Arcturus.priority)

['Deneb', 'M13', 'Vega', 'Polaris', 'Altair', 'Albireo', 'Arcturus', 'Capella']
the priority right now is  1
 these are the obs blocks [<astroplan.scheduling.ObservingBlock (Arcturus, 2025-07-06 19:00:00.000 to 2025-07-06 19:05:00.000) at 0x141c1f1a0>] the priority is changed to  10


In [39]:
from astroplan import Observer, FixedTarget
from astropy.time import Time
from astropy import units as u
from astroplan.constraints import AirmassConstraint, AtNightConstraint, TimeConstraint, AltitudeConstraint
from pyscope.telrun.create_priority_schedule import create_priority_schedule  # replace with your actual import

# 1. Define your targets and (optional) names
targets = [
    FixedTarget.from_name('Deneb'),
    FixedTarget.from_name('M13'),
]


# 2. Create an Observer
observer = Observer.at_site('Rubin')

# 3. Define your time window
start_time      = Time('2025-07-06 19:00')  # UT
end_time        = Time('2025-07-07 19:00')  # UT

# 6 h half‑night sub‑window
half_night_start = Time('2025-07-07 02:00')  # UT
half_night_end   = Time('2025-07-07 08:00')  # UT

# 4. (Optional) per‑target priorities and durations
  # lower number = schedule first
targets = [FixedTarget.from_name('Deneb'), FixedTarget.from_name('M13'), 
                   FixedTarget.from_name('Vega'), 
                   FixedTarget.from_name('Polaris'), 
                   FixedTarget.from_name('Altair'), 
                   FixedTarget.from_name('Albireo'),
                   FixedTarget.from_name('Arcturus'),FixedTarget.from_name('Capella')]
durations = [5*u.minute, 5*u.minute, 5*u.minute, 5*u.minute, 5*u.minute, 5*u.minute, 5*u.minute, 5*u.minute]
constraints = []
configuration = [{'filter': 'B'}] + [{}]*7
                

# 8. Call your function
schedule, scheduler = create_priority_schedule(
    targets=targets,
    observer=observer,
    start_time=start_time,
    end_time=end_time,
    durations=durations,
    constraints=constraints,
    configuration= configuration
    
)

# 9. Inspect the schedule
from astropy.table import Table
schedule.scheduled_blocks
from astroplan.scheduling import TransitionBlock


schedule.to_table()



['Deneb', 'M13', 'Vega', 'Polaris', 'Altair', 'Albireo', 'Arcturus', 'Capella']


target,start time (UTC),end time (UTC),duration (minutes),ra,dec,configuration
str15,str23,str23,float64,str32,str32,object
Arcturus,2025-07-06 19:00:00.000,2025-07-06 19:05:00.000,4.999999999999982,213.915300295,19.182409162,{}
TransitionBlock,2025-07-06 19:05:00.000,2025-07-06 19:06:00.000,0.9999999999999964,,,[]
M13,2025-07-06 22:06:59.999,2025-07-06 22:11:59.999,4.999999999999982,250.423475,36.46131944,{}
TransitionBlock,2025-07-06 22:11:59.999,2025-07-06 22:12:59.999,0.9999999999999964,,,[]
Altair,2025-07-06 23:55:59.999,2025-07-07 00:00:59.999,4.999999999999982,297.695827296,8.868321196,{}
TransitionBlock,2025-07-07 00:00:59.999,2025-07-07 00:01:59.999,0.9999999999999964,,,[]
Vega,2025-07-07 00:11:59.999,2025-07-07 00:16:59.999,4.999999999999982,279.234734787,38.783688956,{}
TransitionBlock,2025-07-07 00:16:59.999,2025-07-07 00:17:59.999,0.9999999999999964,,,[]
Albireo,2025-07-07 00:26:59.999,2025-07-07 00:31:59.999,4.999999999999982,292.68031501,27.95967363,{}


In [40]:
from pyscope.telrun import TelrunOperator

telrun= TelrunOperator(gui=False)



telrun.cfg already exists, skipping
observatory.cfg already exists, skipping
notifications.cfg already exists, skipping
sync.cfg already exists, skipping
/Users/ujjwalsingh/Documents/pyscope/tests/telrun/images already exists, skipping
/Users/ujjwalsingh/Documents/pyscope/tests/telrun/images/autofocus already exists, skipping
/Users/ujjwalsingh/Documents/pyscope/tests/telrun/images/calibrations already exists, skipping
/Users/ujjwalsingh/Documents/pyscope/tests/telrun/images/calibrations/masters already exists, skipping
/Users/ujjwalsingh/Documents/pyscope/tests/telrun/images/raw_archive already exists, skipping
/Users/ujjwalsingh/Documents/pyscope/tests/telrun/images/recenter already exists, skipping
/Users/ujjwalsingh/Documents/pyscope/tests/telrun/images/reduced already exists, skipping
/Users/ujjwalsingh/Documents/pyscope/tests/telrun/logs already exists, skipping
/Users/ujjwalsingh/Documents/pyscope/tests/telrun/schedules already exists, skipping
/Users/ujjwalsingh/Documents/pysco

TelrunException: observatory must be a string representing an observatory config file path or an Observatory object, currently /Users/ujjwalsingh/Documents/pyscope/tests/telrun/config/observatory.cfg, <class 'pathlib.PosixPath'>

# Testing if I can get sunrise and sunset times for observatories and use it as a TimeConstraint

In [ ]:
from astroplan import Observer, FixedTarget, ObservingBlock
from astropy.time import Time
from astropy import units as u
from astroplan.constraints import AirmassConstraint, AtNightConstraint, TimeConstraint, AltitudeConstraint
from astroplan.scheduling import TransitionBlock, Schedule, Transitioner
from pyscope.telrun.bbscheduler import BBScheduler
from astroplan.scheduling import PriorityScheduler
def get_test_constants():
    """Return common constants used across all tests."""
    observer = Observer.at_site('apo')
    start_time = Time('2025-07-06 19:00')  # UT
    end_time = Time('2025-07-07 19:00')  # UT
    half_night_start = Time('2025-07-07 02:00')  # UT
    half_night_end = Time('2025-07-07 08:00')  # UT
    return {
        'observer': observer,
        'start_time': start_time,
        'end_time': end_time,
        'half_night_start': half_night_start,
        'half_night_end': half_night_end
    }
constants = get_test_constants()
observer = constants['observer']  # APO observatory
time = constants['start_time']    # July 6, 2025
    
    # Get sunset and sunrise times
sunset = observer.sun_set_time(time, which='next')
sunrise = observer.sun_rise_time(time, which='next')
print(sunset, " is the sunset time and ", sunrise, " is the sunrise time")

2460863.5904946527  is the sunset time and  2460864.004402679  is the sunrise time


In [3]:
from pyscope.telrun.sch_blocks import get_all_test_blocks

In [76]:
blocks=get_all_test_blocks()
blocks.keys()

dict_keys(['alc', 'xpg1', 'xpgtest'])

Changing gap_time and time_resolution can make scheduling fail

In [32]:
from pyscope.telrun.bbscheduler import BBScheduler
from astroplan import Observer, FixedTarget, ObservingBlock
from astropy.time import Time
from astropy import units as u
from astroplan.constraints import AirmassConstraint, AtNightConstraint, TimeConstraint, AltitudeConstraint
from astroplan.scheduling import TransitionBlock, Schedule, Transitioner
from pyscope.telrun.bbscheduler import BBScheduler
from astroplan.scheduling import PriorityScheduler
apo  = Observer.at_site('subaru')
global_constraints = []
transitioner = Transitioner(slew_rate=1*u.deg/u.second)
# Initialize the priority scheduler with the constraints and transitioner
prior_scheduler = BBScheduler(constraints = global_constraints,
                                    observer = apo,
                                    transitioner = transitioner, gap_time=1*u.minute, time_resolution=1*u.minute)
# Initialize a Schedule object, to contain the new schedule
priority_schedule = Schedule(Time('2025-07-06 23:00'), Time('2025-07-07 01:00'))
all = blocks['xpg1'] + blocks['xpgtest'] + blocks['alc']
# Call the schedule with the observing blocks and schedule to schedule the blocks
schedule = prior_scheduler(all, priority_schedule)

In [33]:
prior_scheduler.get_scheduling_summary()


{'total_blocks': 39,
 'scheduled_blocks': 15,
 'missing_blocks': 24,
 'scheduling_efficiency': 38.46153846153847}

In [34]:
len(all)

39

In [21]:
len(schedule.scheduled_blocks)

58

In [27]:
len(schedule.observing_blocks)

19

In [35]:
schedule.to_table().show_in_notebook()

         interactive tables it is recommended to use dedicated tools like:
         - https://github.com/bloomberg/ipydatagrid
         - https://docs.bokeh.org/en/latest/docs/user_guide/interaction/widgets.html#datatable
         - https://dash.plotly.com/datatable [warnings]


idx,target,start time (UTC),end time (UTC),duration (minutes),ra,dec,configuration
0,1E 0754+39.3,2025-07-06 23:00:00.000,2025-07-06 23:02:00.000,1.999999999999993,119.25,39.3,"{'filter': 'g', 'binning': (2, 2), 'exposure': 60, 'nexp': 2, 'repositioning': (2394, 1597)}"
1,1E 0754+39.3,2025-07-06 23:02:00.000,2025-07-06 23:12:00.000,9.999999999999964,119.25,39.3,"{'filter': 'g', 'binning': (2, 2), 'exposure': 300, 'nexp': 2, 'repositioning': (2394, 1597)}"
2,TransitionBlock,2025-07-06 23:12:00.000,2025-07-06 23:13:00.000,0.9999999999999964,,,[]
3,OJ 287,2025-07-06 23:13:00.000,2025-07-06 23:15:00.000,1.999999999999993,133.0,20.1,"{'filter': 'g', 'binning': (2, 2), 'exposure': 60, 'nexp': 2}"
4,OJ 287,2025-07-06 23:16:00.000,2025-07-06 23:26:00.000,9.999999999999964,133.0,20.1,"{'filter': 'g', 'binning': (2, 2), 'exposure': 300, 'nexp': 2}"
5,TransitionBlock,2025-07-06 23:26:00.000,2025-07-06 23:27:00.000,0.9999999999999964,,,[]
6,1E 0754+39.3,2025-07-06 23:27:00.000,2025-07-06 23:42:00.000,15.000000000000107,119.25,39.3,"{'filter': 'lrg', 'binning': (2, 2), 'exposure': 300, 'nexp': 3, 'repositioning': (2394, 1597)}"
7,TransitionBlock,2025-07-06 23:42:00.000,2025-07-06 23:43:00.000,0.9999999999999964,,,[]
8,OJ 287,2025-07-06 23:43:00.000,2025-07-06 23:53:00.000,9.999999999999964,133.0,20.1,"{'filter': 'lrg', 'binning': (2, 2), 'exposure': 300, 'nexp': 2}"
9,TransitionBlock,2025-07-06 23:53:00.000,2025-07-06 23:54:00.000,0.9999999999999964,,,[]


In [36]:
from pyscope.telrun.sched_ecsv import create_ecsv_table

create_ecsv_table(schedule)


TypeError: 'Schedule' object is not iterable

39

In [88]:
from pyscope.telrun.bbscheduler import BBScheduler
from astroplan import Observer, FixedTarget, ObservingBlock
from astropy.time import Time
from astropy import units as u
from astroplan.constraints import AirmassConstraint, AtNightConstraint, TimeConstraint, AltitudeConstraint
from astroplan.scheduling import TransitionBlock, Schedule, Transitioner
from astroplan.scheduling import PriorityScheduler
apo  = Observer.at_site('subaru')
global_constraints = []
transitioner = Transitioner(slew_rate=1*u.deg/u.second)
# Initialize the priority scheduler with the constraints and transitioner
bb_scheduler_instance = BBScheduler(constraints = global_constraints,
                                    observer = apo,
                                    transitioner = transitioner, gap_time=1*u.minute, time_resolution=1*u.minute)
# Initialize a Schedule object, to contain the new schedule
priority_schedule = Schedule(Time('2025-07-06 23:00'), Time('2025-07-07 01:00'))

# Call the schedule with the observing blocks and schedule to schedule the blocks
schedule = bb_scheduler_instance(blocks['xpg1'], priority_schedule)

In [87]:
bb_scheduler_instance.get_mis
sing_blocks()

[]

In [79]:
prior_scheduler

[<astroplan.scheduling.ObservingBlock (1E 0754+39.3, 2025-07-06 23:00:00.000 to 2025-07-06 23:02:00.000) at 0x1431348c0>,
 <astroplan.scheduling.ObservingBlock (1E 0754+39.3, 2025-07-06 23:02:00.000 to 2025-07-06 23:12:00.000) at 0x143135df0>,
 <astroplan.scheduling.ObservingBlock (1E 0754+39.3, 2025-07-06 23:13:00.000 to 2025-07-06 23:28:00.000) at 0x143134200>,
 <astroplan.scheduling.TransitionBlock (slew_time: 16.56330072137773 s, 2025-07-06 23:28:00.000 to 2025-07-06 23:29:00.000) at 0x14162d6a0>,
 <astroplan.scheduling.ObservingBlock (TON951, 2025-07-06 23:29:00.000 to 2025-07-06 23:31:00.000) at 0x14323acc0>,
 <astroplan.scheduling.ObservingBlock (TON951, 2025-07-06 23:32:00.000 to 2025-07-06 23:42:00.000) at 0x14323b650>,
 <astroplan.scheduling.ObservingBlock (TON951, 2025-07-06 23:43:00.000 to 2025-07-06 23:58:00.000) at 0x143238b00>,
 <astroplan.scheduling.TransitionBlock (slew_time: 51.97075235733448 s, 2025-07-06 23:58:00.000 to 2025-07-06 23:59:00.000) at 0x142c48ce0>,
 <as